# Step 3 – Constructing a new dataset: a research-paper citation graph

**Project:** COMP8240 – Reproducing and Extending Mixup for Node Classification in GNNs
**Goal:** build a *new* node-classification dataset that the paper never used, document how it was built, and test whether the Mixup result transfers to it.

| Part | What it does | Needs GPU? |
|---|---|---|
| 1. Setup | installs, Drive, settings | no |
| 2. API check | one tiny request to OpenAlex and arXiv, so problems show up in the first minute | no |
| 3. **Collection** | snowball sampling of arXiv computer-science papers and their citations from **OpenAlex** | no |
| 4. **Labels** | each paper's **arXiv primary category** (chosen by the authors), from the **arXiv API** | no |
| 5. **Preprocessing** | keep papers in the chosen areas, build citation edges, largest connected component, optional k-NN edges | no |
| 6. **Features** | sentence embeddings of title + abstract (MiniLM) | optional |
| 7. Dataset card | statistics, class balance, homophily, saved files | no |
| 8. **Annotation** | a 100-paper sheet for a manual label check | no |
| 9. **Experiments** | GCN vs Mixup+GCN and GraphConv vs Mixup+GraphConv (same protocol as Table 3) | optional |

**Design choices (from the proposal, Section 5.1)**
* **Nodes** = arXiv computer-science papers; **edges** = citations between them (undirected). Papers are collected by **snowball sampling** from highly cited seed papers so the graph is not too sparse.
* **Labels** = the paper's arXiv *primary category* (e.g. `cs.CV`), which the **authors** chose when submitting. This avoids labels that were themselves predicted from the abstract (label leakage).
* **Features** = fixed pretrained sentence embeddings of title + abstract (dense features suit Mixup better than sparse TF-IDF).
* If the citation graph is too sparse, a **k-NN text-similarity graph** is added as a documented fallback.

Everything downloaded is cached on Google Drive, so re-running never repeats a finished request.

---
## 1. Setup
### 1.1 Install and import

In [ ]:
%pip install -q torch_geometric sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.0 MB/s eta 0:00:00


In [ ]:
import os, os.path as osp, re, json, time, math, random, hashlib, platform, warnings
import xml.etree.ElementTree as ET
from types import SimpleNamespace
from collections import Counter

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch_geometric
from torch_geometric.nn import GATConv

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('python', platform.python_version(), '| torch', torch.__version__, '| pyg', torch_geometric.__version__, '| device', DEVICE)

python 3.13.15 | torch 2.11.0+cu130 | pyg 2.8.0.post1 | device cuda


### 1.2 Settings
* `AREAS` – the arXiv primary categories that become the classes. Each has a search phrase used **only to pick seed papers**; labels always come from arXiv, never from these phrases.
* `SEEDS_PER_AREA`, `REFS_PER_PAPER`, `CITING_PER_PAPER`, `HOPS` – size of the snowball sample.
* `MAX_NODES` – upper limit on the final graph size (the largest connected component is kept; if it is larger, the best-connected papers are kept).
* `MAILTO` – optional: put your e-mail here to use OpenAlex's faster "polite pool". Leave empty if you prefer.

In [ ]:
USE_DRIVE = True
MAILTO = ''                      # optional, e.g. 'you@example.com'

AREAS = {                        # arXiv primary category -> (class name, seed search phrase)
    'cs.CL': ('NLP',              'natural language processing language model'),
    'cs.CV': ('Computer vision',  'image recognition convolutional network'),
    'cs.LG': ('Machine learning', 'deep learning optimization generalization'),
    'cs.RO': ('Robotics',         'robot navigation manipulation control'),
    'cs.CR': ('Security',         'adversarial attack privacy security'),
    'cs.IR': ('Info. retrieval',  'recommender system retrieval ranking'),
}
YEARS = (2016, 2023)             # publication years for seed papers
SEED_MIN_CITES = 50              # seeds must be reasonably well cited (connected)
SEEDS_PER_AREA = 25
REFS_PER_PAPER = 60              # references followed per paper
CITING_PER_PAPER = 40            # most-cited citing papers followed per paper
HOPS = 1                         # 1 hop from the seeds is usually enough for ~3k nodes
MAX_NODES = 4000
KNN_K = 5                        # k for the optional text-similarity edges
ADD_KNN_IF_ISOLATED_OVER = 0.15  # add k-NN edges to the 'all nodes' variant if >15% of nodes have no citation edge

ARXIV_SOURCE = 'S4306400194'     # OpenAlex source id of arXiv (checked in Section 2)
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/mixup_reproduction/paper_graph'
else:
    ROOT = './paper_graph'
CACHE = osp.join(ROOT, 'api_cache')
os.makedirs(CACHE, exist_ok=True)
print('Saving to', ROOT)

Mounted at /content/drive
Saving to /content/drive/MyDrive/mixup_reproduction/paper_graph


### 1.3 API helpers (with caching and retries)
* **OpenAlex** (`api.openalex.org`) – free, no key; gives each paper's references, the papers citing it, and its locations (from which the arXiv id is read).
* **arXiv API** (`export.arxiv.org`) – gives title, abstract and **primary category**. arXiv asks for at most one request every 3 seconds, which the helper respects.

Every response is stored in `api_cache/` on Drive, keyed by the request, so a re-run is instant.

In [ ]:
SESSION = requests.Session()
SESSION.headers['User-Agent'] = 'COMP8240-mixup-reproduction (student project)'
SELECT = 'id,display_name,publication_year,cited_by_count,referenced_works,locations,primary_topic,doi'
_last_arxiv_call = [0.0]


def _cache_path(tag, key):
    return osp.join(CACHE, f'{tag}_{hashlib.md5(key.encode()).hexdigest()}.json')


def _get_json(url, params, tag, max_tries=6):
    key = url + '?' + '&'.join(f'{k}={params[k]}' for k in sorted(params))
    path = _cache_path(tag, key)
    if osp.exists(path):
        with open(path) as f:
            return json.load(f)
    for attempt in range(max_tries):
        try:
            r = SESSION.get(url, params=params, timeout=60)
            if r.status_code == 200:
                data = r.json()
                with open(path, 'w') as f:
                    json.dump(data, f)
                return data
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(f'{r.status_code} for {r.url}: {r.text[:300]}')
        except requests.RequestException as e:
            time.sleep(2 ** attempt)
            last = e
    raise RuntimeError(f'giving up on {url} {params}')


def openalex(params):
    params = dict(params)
    if MAILTO:
        params['mailto'] = MAILTO
    return _get_json('https://api.openalex.org/works', params, 'oa')


def short_id(w):
    return w.rsplit('/', 1)[-1] if isinstance(w, str) else w['id'].rsplit('/', 1)[-1]


ARXIV_ID_RE = re.compile(r'arxiv\.org/(?:abs|pdf)/(\d{4}\.\d{4,5}|[a-z\-]+(?:\.[A-Za-z]{2})?/\d{7})', re.I)
ARXIV_DOI_RE = re.compile(r'10\.48550/arxiv\.(\d{4}\.\d{4,5}|[a-z\-]+(?:\.[A-Za-z]{2})?/\d{7})', re.I)


def arxiv_id_of(work):
    """arXiv identifier of an OpenAlex work (version suffix removed), or None."""
    for loc in work.get('locations') or []:
        for field in ('landing_page_url', 'pdf_url'):
            m = ARXIV_ID_RE.search(loc.get(field) or '')
            if m:
                return m.group(1)
    m = ARXIV_DOI_RE.search(work.get('doi') or '')
    return m.group(1) if m else None


ATOM = '{http://www.w3.org/2005/Atom}'
ARX = '{http://arxiv.org/schemas/atom}'


def arxiv_metadata(ids, batch=100):
    """Title, abstract, primary category and all categories for a list of arXiv ids."""
    out = {}
    ids = sorted(set(ids))
    for i in tqdm(range(0, len(ids), batch), desc='arXiv API'):
        chunk = ids[i:i + batch]
        key = ','.join(chunk)
        path = _cache_path('arxiv', key)
        if osp.exists(path):
            with open(path) as f:
                xml = json.load(f)['xml']
        else:
            wait = 3.1 - (time.time() - _last_arxiv_call[0])
            if wait > 0:
                time.sleep(wait)
            for attempt in range(6):
                r = SESSION.get('https://export.arxiv.org/api/query',
                                params={'id_list': key, 'max_results': len(chunk)}, timeout=120)
                _last_arxiv_call[0] = time.time()
                if r.status_code == 200:
                    break
                time.sleep(3 * 2 ** attempt)
            r.raise_for_status()
            xml = r.text
            with open(path, 'w') as f:
                json.dump({'xml': xml}, f)
        root = ET.fromstring(xml)
        for e in root.findall(ATOM + 'entry'):
            eid = (e.findtext(ATOM + 'id') or '')
            m = re.search(r'arxiv\.org/abs/(.+?)(v\d+)?$', eid)
            prim = e.find(ARX + 'primary_category')
            if not m or prim is None:
                continue
            out[m.group(1)] = dict(
                title=' '.join((e.findtext(ATOM + 'title') or '').split()),
                abstract=' '.join((e.findtext(ATOM + 'summary') or '').split()),
                primary=prim.get('term'),
                categories=[c.get('term') for c in e.findall(ATOM + 'category')],
                published=(e.findtext(ATOM + 'published') or '')[:10])
    return out

print('helpers ready')

helpers ready


---
## 2. API check (run this first – takes a few seconds)
A few OpenAlex requests and one arXiv request. **Expected:** a count above 0, an arXiv id for the example paper, `lookup by id` = 1, a large `cites` count, and a primary category such as `cs.CL`. If something fails here, copy the output and ask for help before running the rest.

In [ ]:
chk = openalex({'search': 'attention is all you need', 'per-page': 3, 'select': SELECT,
                'filter': f'locations.source.id:{ARXIV_SOURCE}'})
print('OpenAlex results for the test query:', chk['meta']['count'])
ARX_F = f',locations.source.id:{ARXIV_SOURCE}'     # appended to filters to ask OpenAlex for arXiv papers only
if chk['meta']['count'] == 0:
    print('!! The arXiv source filter returned nothing -> collection will run without it '
          '(non-arXiv papers are dropped afterwards, so it is slower but still correct).')
    ARX_F = ''
    chk = openalex({'search': 'attention is all you need', 'per-page': 3, 'select': SELECT})
for w in chk['results']:
    print(' ', short_id(w), '|', w['display_name'][:60], '| arXiv id:', arxiv_id_of(w),
          '| refs:', len(w.get('referenced_works') or []))
ids = [a for a in (arxiv_id_of(w) for w in chk['results']) if a]
if chk['results']:
    w0 = short_id(chk['results'][0])
    by_id = openalex({'filter': f'openalex:{w0}', 'per-page': 5, 'select': SELECT})
    citing = openalex({'filter': f'cites:{w0}{ARX_F}', 'per-page': 5, 'select': SELECT})
    print(f'lookup by id (openalex: filter): {by_id["meta"]["count"]} result(s)   <- should be 1')
    print(f'papers citing it on arXiv (cites: filter): {citing["meta"]["count"]}   <- should be large')
meta = arxiv_metadata(ids[:1]) if ids else {}
print('arXiv API:', {k: (v['primary'], v['title'][:50]) for k, v in meta.items()} or 'no arXiv id found in the test results')

OpenAlex results for the test query: 150241
  W2626778328 | Attention Is All You Need | arXiv id: 1706.03762 | refs: 28
  W3126721948 | Is Space-Time Attention All You Need for Video Understanding | arXiv id: 2102.05095 | refs: 58
  W3155713635 | Not All Attention Is All You Need | arXiv id: 2104.04692 | refs: 51
lookup by id (openalex: filter): 1 result(s)   <- should be 1
papers citing it on arXiv (cites: filter): 11935   <- should be large


arXiv API:   0%|          | 0/1 [00:00<?, ?it/s]

arXiv API: {'1706.03762': ('cs.CL', 'Attention Is All You Need')}


---
## 3. Collection: snowball sampling from seed papers (OpenAlex)
1. **Seeds:** for every area, the `SEEDS_PER_AREA` most-cited arXiv papers matching the area's search phrase (years `YEARS`, at least `SEED_MIN_CITES` citations).
2. **Snowball (1 hop):** from every seed, follow up to `REFS_PER_PAPER` of its **references** and its `CITING_PER_PAPER` most-cited **citing papers**.
3. Keep only works that are on arXiv (so they can be labelled).

Typical run: ~300–600 OpenAlex requests, ~5–15 minutes the first time (instant when cached).

In [ ]:
works = {}                                   # OpenAlex id -> work record


def add_works(results):
    new = []
    for w in results:
        wid = short_id(w)
        if wid not in works and arxiv_id_of(w):
            works[wid] = w
            new.append(wid)
    return new


# ---- 1. seeds ----
seed_ids = []
for cat, (name, phrase) in AREAS.items():
    r = openalex({'search': phrase, 'sort': 'cited_by_count:desc', 'per-page': SEEDS_PER_AREA, 'select': SELECT,
                  'filter': f'publication_year:{YEARS[0]}-{YEARS[1]},cited_by_count:>{SEED_MIN_CITES}{ARX_F}'})
    got = add_works(r['results'])
    seed_ids += got
    print(f'{cat:6s} {name:18s} seeds: {len(got)}')
print('total seeds:', len(seed_ids))

# ---- 2. snowball ----
frontier = list(seed_ids)
for hop in range(HOPS):
    ref_ids, nxt = set(), []
    for wid in tqdm(frontier, desc=f'hop {hop + 1}: citing papers'):
        ref_ids.update(short_id(x) for x in (works[wid].get('referenced_works') or [])[:REFS_PER_PAPER])
        r = openalex({'filter': f'cites:{wid}{ARX_F}', 'sort': 'cited_by_count:desc',
                      'per-page': CITING_PER_PAPER, 'select': SELECT})
        nxt += add_works(r['results'])
    todo = sorted(ref_ids - set(works))
    for i in tqdm(range(0, len(todo), 50), desc=f'hop {hop + 1}: references'):
        r = openalex({'filter': 'openalex:' + '|'.join(todo[i:i + 50]), 'per-page': 50, 'select': SELECT})
        nxt += add_works(r['results'])
    frontier = nxt
    print(f'after hop {hop + 1}: {len(works)} arXiv works collected')

# merge OpenAlex records that point to the same arXiv paper
by_arxiv = {}
for wid, w in works.items():
    a = arxiv_id_of(w)
    if a in by_arxiv:
        by_arxiv[a]['refs'] |= set(short_id(x) for x in w.get('referenced_works') or [])
        by_arxiv[a]['oa_ids'].add(wid)
    else:
        by_arxiv[a] = dict(oa_ids={wid}, refs=set(short_id(x) for x in w.get('referenced_works') or []),
                           year=w.get('publication_year'), cites=w.get('cited_by_count'),
                           oa_subfield=((w.get('primary_topic') or {}).get('subfield') or {}).get('display_name'),
                           seed=wid in seed_ids)
print('unique arXiv papers:', len(by_arxiv))
collection_log = dict(seeds=len(seed_ids), openalex_works=len(works), unique_arxiv=len(by_arxiv))

cs.CL  NLP                seeds: 25
cs.CV  Computer vision    seeds: 13
cs.LG  Machine learning   seeds: 16
cs.RO  Robotics           seeds: 25
cs.CR  Security           seeds: 20
cs.IR  Info. retrieval    seeds: 22
total seeds: 121


hop 1: citing papers:   0%|          | 0/121 [00:00<?, ?it/s]

hop 1: references:   0%|          | 0/85 [00:00<?, ?it/s]

after hop 1: 4652 arXiv works collected
unique arXiv papers: 4605


---
## 4. Labels and text from the arXiv API
For every collected paper: title, abstract, **primary category** (the label) and all cross-listed categories. ~1 request per 100 papers, 3 s apart (≈ 1–3 minutes).

In [ ]:
meta = arxiv_metadata(list(by_arxiv))
print(f'arXiv metadata found for {len(meta)} / {len(by_arxiv)} papers')
prim = Counter(m['primary'] for m in meta.values())
print('most common primary categories:', prim.most_common(12))
collection_log['arxiv_metadata'] = len(meta)

arXiv API:   0%|          | 0/47 [00:00<?, ?it/s]

arXiv metadata found for 4605 / 4605 papers
most common primary categories: [('cs.CV', 1009), ('cs.LG', 846), ('cs.CL', 533), ('cs.CR', 212), ('cs.AI', 196), ('eess.IV', 158), ('cs.RO', 157), ('stat.ML', 135), ('quant-ph', 97), ('cs.IR', 90), ('cs.CY', 89), ('eess.SP', 80)]


---
## 5. Preprocessing
1. Keep papers whose primary category is one of `AREAS`, with a non-empty abstract; drop duplicates.
2. Citation edges = pairs of kept papers where one cites the other (made undirected, no self-loops).
3. **Main graph:** the largest connected component (LCC) of the citation graph, capped at `MAX_NODES` (best-connected papers kept).
4. **Fallback variant:** all kept papers + k-NN text-similarity edges, built only if many papers have no citation edge (Section 6.1).

In [ ]:
rows, seen_titles = [], set()
for a, info in by_arxiv.items():
    m = meta.get(a)
    if not m or m['primary'] not in AREAS or len(m['abstract']) < 50:
        continue
    t = re.sub(r'\W+', ' ', m['title'].lower()).strip()
    if t in seen_titles:                     # duplicate title (e.g. re-submission)
        continue
    seen_titles.add(t)
    rows.append(dict(arxiv_id=a, title=m['title'], abstract=m['abstract'], label_code=m['primary'],
                     label=AREAS[m['primary']][0], cross_listed=len(m['categories']) - 1,
                     year=int(m['published'][:4]) if m['published'][:4].isdigit() else info['year'],
                     cites=info['cites'], oa_subfield=info['oa_subfield'], seed=info['seed']))
papers = pd.DataFrame(rows)
oa_to_arxiv = {oid: a for a, info in by_arxiv.items() for oid in info['oa_ids']}
index = {a: i for i, a in enumerate(papers.arxiv_id)}

edges = set()
for a in papers.arxiv_id:
    for ref in by_arxiv[a]['refs']:
        b = oa_to_arxiv.get(ref)
        if b in index and b != a:
            i, j = index[a], index[b]
            edges.add((min(i, j), max(i, j)))
edges = np.array(sorted(edges), dtype=np.int64).reshape(-1, 2)
print(f'kept papers: {len(papers)} | citation edges among them: {len(edges)}')


def components(n, e):
    parent = np.arange(n)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j in e:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    return np.array([find(i) for i in range(n)])


comp = components(len(papers), edges)
sizes = Counter(comp)
lcc_root = sizes.most_common(1)[0][0]
in_lcc = comp == lcc_root
deg_all = np.bincount(edges.ravel(), minlength=len(papers)) if len(edges) else np.zeros(len(papers), int)
print(f'connected components: {len(sizes)} | largest: {in_lcc.sum()} papers | '
      f'isolated papers: {(deg_all == 0).sum()} ({100 * (deg_all == 0).mean():.1f}%)')

keep = np.where(in_lcc)[0]
if len(keep) > MAX_NODES:                    # cap size: keep best-connected papers, then re-take the LCC
    keep = keep[np.argsort(-deg_all[keep])[:MAX_NODES]]
    sub = np.zeros(len(papers), bool); sub[keep] = True
    e2 = edges[sub[edges[:, 0]] & sub[edges[:, 1]]]
    c2 = components(len(papers), e2)
    root = Counter(c2[keep]).most_common(1)[0][0]
    keep = keep[c2[keep] == root]
remap = -np.ones(len(papers), dtype=np.int64); remap[keep] = np.arange(len(keep))
lcc_edges = remap[edges]
lcc_edges = lcc_edges[(lcc_edges >= 0).all(1)]
lcc_papers = papers.iloc[keep].reset_index(drop=True)
print(f'MAIN GRAPH (citation LCC): {len(lcc_papers)} papers, {len(lcc_edges)} undirected edges')

kept papers: 2847 | citation edges among them: 19936
connected components: 35 | largest: 2813 papers | isolated papers: 34 (1.2%)
MAIN GRAPH (citation LCC): 2813 papers, 19936 undirected edges


---
## 6. Node features: sentence embeddings of title + abstract
`all-MiniLM-L6-v2` gives a 384-dimensional vector per paper (≈ 1 min on GPU, a few minutes on CPU). If the model cannot be downloaded, a TF-IDF + SVD fallback with the same size is used and the notebook says so. The category names are **not** part of the text, so the label is not leaked into the features.

In [ ]:
texts = (papers.title + '. ' + papers.abstract).tolist()
try:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer(EMBED_MODEL, device=str(DEVICE))
    X_all = enc.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    feature_method = EMBED_MODEL
except Exception as e:
    print('sentence-transformers unavailable (', e, ') -> TF-IDF + SVD fallback')
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    tf = TfidfVectorizer(max_features=20000, stop_words='english', sublinear_tf=True).fit_transform(texts)
    X_all = TruncatedSVD(384, random_state=0).fit_transform(tf)
    X_all /= np.linalg.norm(X_all, axis=1, keepdims=True) + 1e-12
    feature_method = 'tfidf-svd-384'
X_all = X_all.astype(np.float32)
print('features:', X_all.shape, 'via', feature_method)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

features: (2847, 384) via sentence-transformers/all-MiniLM-L6-v2


### 6.1 Optional fallback variant: all papers + k-NN text-similarity edges
Built only if more than `ADD_KNN_IF_ISOLATED_OVER` of the kept papers have no citation edge. Each paper is linked to its `KNN_K` most similar papers (cosine similarity of the embeddings), in addition to its citations. Because these edges come from the same text as the features, results on this variant are reported separately from the main citation graph.

In [ ]:
knn_variant = None
iso_frac = float((deg_all == 0).mean())
if iso_frac > ADD_KNN_IF_ISOLATED_OVER:
    Xt = torch.from_numpy(X_all)
    knn = set()
    for s in range(0, len(Xt), 2048):
        sim = Xt[s:s + 2048] @ Xt.T
        sim[torch.arange(sim.size(0)), torch.arange(s, s + sim.size(0))] = -1
        nb = sim.topk(KNN_K, dim=1).indices.numpy()
        for r_, row in enumerate(nb):
            i = s + r_
            for j in row:
                knn.add((min(i, j), max(i, j)))
    both = set(map(tuple, edges.tolist())) | knn
    knn_variant = np.array(sorted(both), dtype=np.int64)
    print(f'{100 * iso_frac:.1f}% isolated -> k-NN variant built: {len(papers)} papers, {len(knn_variant)} edges '
          f'({len(knn - set(map(tuple, edges.tolist())))} added by k-NN)')
else:
    print(f'only {100 * iso_frac:.1f}% isolated papers -> citation graph is dense enough; no k-NN variant needed')

only 1.2% isolated papers -> citation graph is dense enough; no k-NN variant needed


---
## 7. Dataset card: statistics, class balance, homophily
**Edge homophily** = share of edges that connect two papers with the same label. Cora is ≈ 0.81; GNNs work best when this is high. It is a key property to report for a new graph dataset.

In [ ]:
def graph_stats(name, df, e):
    n = len(df)
    y = df.label.values
    deg = np.bincount(e.ravel(), minlength=n) if len(e) else np.zeros(n, int)
    return dict(graph=name, nodes=n, undirected_edges=len(e), avg_degree=round(2 * len(e) / max(n, 1), 2),
                isolated=int((deg == 0).sum()), classes=df.label.nunique(), features=X_all.shape[1],
                edge_homophily=round(float((y[e[:, 0]] == y[e[:, 1]]).mean()), 3) if len(e) else np.nan,
                largest_class=f'{100 * df.label.value_counts(normalize=True).max():.1f}%',
                years=f'{int(df.year.min())}–{int(df.year.max())}')


stats_rows = [graph_stats('citation LCC (main)', lcc_papers, lcc_edges)]
if knn_variant is not None:
    stats_rows.append(graph_stats('all papers + k-NN', papers, knn_variant))
dataset_stats = pd.DataFrame(stats_rows).set_index('graph')
display(dataset_stats)

class_table = (lcc_papers.groupby(['label', 'label_code'])
                         .agg(papers=('arxiv_id', 'size'), seeds=('seed', 'sum'),
                              median_year=('year', 'median'), cross_listed_share=('cross_listed', lambda s: round((s > 0).mean(), 2)))
                         .sort_values('papers', ascending=False))
class_table['share'] = (100 * class_table.papers / class_table.papers.sum()).round(1).astype(str) + '%'
display(class_table)
print('collection log:', collection_log)

,nodes,undirected_edges,avg_degree,isolated,classes,features,edge_homophily,largest_class,years
graph,,,,,,,,,
citation LCC (main),2813,19936,14.17,0,6,384,0.636,35.6%,2008–2026


,,papers,seeds,median_year,cross_listed_share,share
label,label_code,,,,,
Computer vision,cs.CV,1002,29,2020.0,0.46,35.6%
Machine learning,cs.LG,836,25,2020.0,0.84,29.7%
NLP,cs.CL,531,22,2020.0,0.53,18.9%
Security,cs.CR,205,7,2020.0,0.83,7.3%
Robotics,cs.RO,154,4,2020.0,0.73,5.5%
Info. retrieval,cs.IR,85,1,2021.0,0.69,3.0%


collection log: {'seeds': 121, 'openalex_works': 4652, 'unique_arxiv': 4605, 'arxiv_metadata': 4605}


### 7.1 Save the dataset
* `paper_graph.pt` – PyTorch file with features, labels, edges (main graph and, if built, the k-NN variant).
* `papers.csv` – one row per paper (arXiv id, title, label, year, citations, degree) for the report and for the planned LLM-as-a-Judge analysis.
* `edges.csv`, `dataset_card.json` – edges and all statistics and settings.

In [ ]:
classes = sorted(papers.label.unique())
cls_idx = {c: i for i, c in enumerate(classes)}
lcc_rows = papers.reset_index().set_index('arxiv_id').loc[lcc_papers.arxiv_id, 'index'].values
data_main = dict(x=torch.from_numpy(X_all[lcc_rows]), y=torch.tensor([cls_idx[c] for c in lcc_papers.label]),
                 edge_index=torch.from_numpy(lcc_edges.T.copy()), classes=classes,
                 arxiv_ids=lcc_papers.arxiv_id.tolist())
save = dict(main=data_main, feature_method=feature_method)
if knn_variant is not None:
    save['knn'] = dict(x=torch.from_numpy(X_all), y=torch.tensor([cls_idx.get(c, -1) for c in papers.label]),
                       edge_index=torch.from_numpy(knn_variant.T.copy()), classes=classes,
                       arxiv_ids=papers.arxiv_id.tolist())
torch.save(save, osp.join(ROOT, 'paper_graph.pt'))
out = lcc_papers.drop(columns=['abstract']).copy()
out['degree'] = np.bincount(lcc_edges.ravel(), minlength=len(lcc_papers))
out.to_csv(osp.join(ROOT, 'papers.csv'), index=False)
pd.DataFrame(lcc_edges, columns=['source', 'target']).to_csv(osp.join(ROOT, 'edges.csv'), index=False)
card = dict(settings=dict(areas={k: v[0] for k, v in AREAS.items()}, years=YEARS, seed_min_cites=SEED_MIN_CITES,
                          seeds_per_area=SEEDS_PER_AREA, refs_per_paper=REFS_PER_PAPER, citing_per_paper=CITING_PER_PAPER,
                          hops=HOPS, max_nodes=MAX_NODES, knn_k=KNN_K, features=feature_method),
            collection=collection_log, stats=dataset_stats.reset_index().to_dict('records'),
            classes=class_table.reset_index().to_dict('records'), built=time.strftime('%Y-%m-%d %H:%M'))
with open(osp.join(ROOT, 'dataset_card.json'), 'w') as f:
    json.dump(card, f, indent=2, default=str)
print('saved:', sorted(os.listdir(ROOT)))

saved: ['annotation_sheet.csv', 'annotation_sheet.gsheet', 'annotation_sheet_filled.csv', 'api_cache', 'dataset_card.json', 'edges.csv', 'paper_graph.pt', 'paper_graph_mixup_gain.csv', 'paper_graph_results.jsonl', 'paper_graph_results_summary.csv', 'papers.csv']


---
## 8. Annotation: manual label check (≈ 1 hour of your time)
arXiv categories are chosen by authors, but some papers are cross-listed or borderline. A **stratified sample of 100 papers** (proportional to class size, at least 5 per class) is written to `annotation_sheet.csv`.

**How to annotate**
1. Open `annotation_sheet.csv` from Drive in Google Sheets.
2. For each row, read the title and abstract and fill **`human_label_ok`** with `y` (the arXiv label fits) or `n` (it does not). If `n`, write the better class in **`human_label`** (one of the class names).
3. File → Download → CSV, and save it into the same Drive folder as **`annotation_sheet_filled.csv`**.
4. Run the next cell to get the agreement rate and the disagreements.

The OpenAlex *subfield* (assigned automatically by OpenAlex) is shown as a second opinion only.

In [ ]:
ANN_N = 100
rng = np.random.RandomState(0)
share = lcc_papers.label.value_counts(normalize=True)
quota = (share * ANN_N).round().astype(int).clip(lower=5)
sample = pd.concat([lcc_papers[lcc_papers.label == c].sample(min(q, (lcc_papers.label == c).sum()), random_state=rng)
                    for c, q in quota.items()])
sheet = sample[['arxiv_id', 'title', 'abstract', 'label', 'oa_subfield']].copy()
sheet['abstract'] = sheet.abstract.str.slice(0, 600)
sheet['url'] = 'https://arxiv.org/abs/' + sheet.arxiv_id
sheet['human_label_ok'] = ''
sheet['human_label'] = ''
sheet['notes'] = ''
sheet = sheet.sample(frac=1, random_state=1).reset_index(drop=True)          # shuffle so classes are mixed
sheet.to_csv(osp.join(ROOT, 'annotation_sheet.csv'), index=False)
print(f'annotation_sheet.csv written: {len(sheet)} papers; class names allowed in human_label: {classes}')
display(sheet[['arxiv_id', 'title', 'label']].head(8))

annotation_sheet.csv written: 102 papers; class names allowed in human_label: ['Computer vision', 'Info. retrieval', 'Machine learning', 'NLP', 'Robotics', 'Security']


,arxiv_id,title,label
0,1707.08101,Learning to Singulate Objects using a Push Pro...,Robotics
1,1908.10084,Sentence-BERT: Sentence Embeddings using Siame...,NLP
2,2103.17249,StyleCLIP: Text-Driven Manipulation of StyleGA...,Computer vision
3,2309.05273,Formalizing Multimedia Recommendation through ...,Info. retrieval
4,2203.05222,Similarity-based Label Inference Attack agains...,Machine learning
5,1702.07464,Deep Models Under the GAN: Information Leakage...,Security
6,1511.04599,DeepFool: a simple and accurate method to fool...,Machine learning
7,2306.06371,A Comprehensive Review of State-of-The-Art Met...,NLP


In [ ]:
print('Looking in:', ROOT)
print([f for f in sorted(os.listdir(ROOT)) if f.endswith('.csv')])

Looking in: /content/drive/MyDrive/mixup_reproduction/paper_graph
['annotation_sheet.csv', 'annotation_sheet_filled.csv', 'edges.csv', 'paper_graph_mixup_gain.csv', 'paper_graph_results_summary.csv', 'papers.csv']


### 8.1 Agreement (run after you have filled in the sheet)

In [ ]:
filled_path = osp.join(ROOT, 'annotation_sheet_filled.csv')
if osp.exists(filled_path):
    ann = pd.read_csv(filled_path)
    ann = ann[ann.human_label_ok.astype(str).str.strip().str.lower().isin(['y', 'n'])]
    ok = ann.human_label_ok.str.strip().str.lower() == 'y'
    print(f'annotated: {len(ann)} papers | arXiv label judged correct: {ok.sum()} ({100 * ok.mean():.1f}%)')
    display(ann.assign(ok=ok).groupby('label').ok.agg(['mean', 'size']).rename(columns={'mean': 'agreement'}).round(2))
    display(ann[~ok][['arxiv_id', 'title', 'label', 'human_label', 'notes']])
else:
    print('No filled sheet yet – annotate annotation_sheet.csv first (see instructions above).')

annotated: 50 papers | arXiv label judged correct: 48 (96.0%)


,agreement,size
label,,
Computer vision,1.00,12
Info. retrieval,1.00,2
Machine learning,0.89,18
NLP,1.00,11
Robotics,1.00,3
Security,1.00,4


,arxiv_id,title,label,human_label,notes
4,2203.05222,Similarity-based Label Inference Attack agains...,Machine learning,Security,Borderline :privacy attack on ML
34,2201.10528,Explainability in Music Recommender Systems,Machine learning,Info.retrieval,"""recommender system filed under ML""."


---
## 9. Experiments: does the Mixup result transfer to the new graph?
Same protocol and the **same code** as Table 3 of the reproduction notebook (copied below unchanged): random 60/20/20 splits, `NEW_SPLITS` × `NEW_SEEDS` runs, test accuracy at the best-validation epoch.

Methods, chosen to test the main reproduction finding:
* **GCN vs Mixup+GCN** (α = 1) – the paper's described setting (no gain on the citation datasets).
* **GraphConv vs Mixup+GraphConv** (α = 4) – the authors' released setting (gain on Cora/Pubmed).

Runs are cached in `paper_graph_results.jsonl`. With the defaults: 4 methods × 2 depths × 20 runs = 160 short runs (≈ 10 min on GPU, ≈ 45–60 min on CPU).

In [ ]:
T3_CONV = 'gcn'
MIXUP_ALPHA = 1.0
DROPEDGE_P = 0.3
T3_HP = dict(hidden=256, dropout=0.4, lr=1e-3, weight_decay=0.0, epochs=300)
GAT_HP = dict(hidden=8, heads=8, dropout=0.6, lr=5e-3, weight_decay=5e-4, epochs=300)
GRAPH_VARIANT = 'main'                 # 'main' (citation LCC) or 'knn' (only if it was built)
NEW_METHODS = ['GCN', 'Mixup+GCN', 'GraphConv', 'Mixup+GraphConv a4']
NEW_SPLITS, NEW_SEEDS, NEW_DEPTHS = 10, 2, [2, 3]

In [ ]:
# (copied unchanged from the reproduction notebook)
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def edge_index_to_csr(edge_index, num_nodes):
    """CSR with rows = target nodes (edge_index[1]) and columns = source nodes (edge_index[0]).
    Self-loops and duplicate edges are removed."""
    src, tgt = edge_index[0].long(), edge_index[1].long()
    key = tgt * num_nodes + src
    key = torch.unique(key[src != tgt])          # sorted by (target, source), no duplicates
    tgt, src = key // num_nodes, key % num_nodes
    rowptr = torch.zeros(num_nodes + 1, dtype=torch.long)
    rowptr[1:] = torch.cumsum(torch.bincount(tgt, minlength=num_nodes), 0)
    return rowptr, src


def rowptr_from_rows(rows, n):
    rowptr = torch.zeros(n + 1, dtype=torch.long)
    rowptr[1:] = torch.cumsum(torch.bincount(rows, minlength=n), 0)
    return rowptr


def csr_rows(rowptr):
    return torch.repeat_interleave(torch.arange(rowptr.numel() - 1), rowptr.diff())


def agg_weights(kind, row, col, n, deg=None):
    """Neighbour edge weights and per-node self weights for one aggregation scheme."""
    if deg is None:
        deg = torch.bincount(row, minlength=n)
    deg = deg.float()
    if kind == 'gcn':
        dinv = (deg + 1.0).rsqrt()
        return dinv[row] * dinv[col], 1.0 / (deg + 1.0)
    if kind == 'graphconv':
        return 1.0 / deg[row].clamp(min=1.0), torch.zeros(n)
    raise ValueError(kind)


class GraphOps:
    """Sparse neighbour-aggregation operator  h -> A @ h  plus per-node self weights."""
    def __init__(self, rowptr, col, val, n, self_w, device, max_nnz=50_000_000):
        self.n, self.device = n, device
        self.self_w = self_w.to(device)
        if col.numel() <= max_nnz:          # small enough: one COO matrix on the device (supports autograd)
            idx = torch.stack([csr_rows(rowptr), col])
            self.A = torch.sparse_coo_tensor(idx, val, (n, n)).coalesce().to(device)
            self.blocks = None
        else:                               # huge graph: keep CSR on the CPU and multiply in row blocks (inference only)
            self.A = None
            self.rowptr, self.col, self.val = rowptr, col, val
            self.blocks, s = [], 0
            while s < n:
                e = int(torch.searchsorted(rowptr, rowptr[s] + max_nnz, right=True)) - 1
                e = min(max(e, s + 1), n)
                self.blocks.append((s, e))
                s = e

    def agg(self, h):
        if self.A is not None:
            return torch.sparse.mm(self.A, h)
        out = torch.empty(self.n, h.size(1), device=h.device, dtype=h.dtype)
        for s, e in self.blocks:
            p0, p1 = int(self.rowptr[s]), int(self.rowptr[e])
            crow = (self.rowptr[s:e + 1] - p0).to(h.device)
            blk = torch.sparse_csr_tensor(crow, self.col[p0:p1].to(h.device), self.val[p0:p1].to(h.device),
                                          (e - s, self.n))
            out[s:e] = blk @ h
        return out


def make_graph(rowptr, col, n, kind, device, deg=None, edge_scale=None):
    val, self_w = agg_weights(kind, csr_rows(rowptr), col, n, deg)
    if edge_scale is not None:
        val = val * edge_scale
    return GraphOps(rowptr, col, val.float(), n, self_w.float(), device)


def drop_edge_undirected(edge_index, p):
    """DropEdge (Rong et al., 2020): remove a random fraction p of undirected edges."""
    src, dst = edge_index
    und = edge_index[:, src < dst]
    und = und[:, torch.rand(und.size(1)) >= p]
    return torch.cat([und, und.flip(0)], dim=1)


@torch.no_grad()
def micro_f1(logits, y, multilabel):
    """F1-micro. For single-label tasks this equals accuracy."""
    if not multilabel:
        return (logits.argmax(1) == y).float().mean().item()
    pred, y = logits > 0, y.bool()
    tp = (pred & y).sum().item()
    fp = (pred & ~y).sum().item()
    fn = (~pred & y).sum().item()
    return 2 * tp / max(2 * tp + fp + fn, 1)

print('utilities ready')

utilities ready


In [ ]:
# (copied unchanged from the reproduction notebook)
class MixConv(nn.Module):
    """One graph-convolution layer split into a neighbour part and a self (centre) part."""
    def __init__(self, d_in, d_out, kind):
        super().__init__()
        self.kind = kind
        self.lin = nn.Linear(d_in, d_out, bias=False)          # neighbour transform (shared with self term for GCN)
        if kind == 'gcn':
            self.bias = nn.Parameter(torch.zeros(d_out))
            nn.init.xavier_uniform_(self.lin.weight)
        else:
            self.root = nn.Linear(d_in, d_out, bias=True)      # separate root transform (GraphConv / SAGE)

    def forward(self, G, h):                                    # ordinary layer
        if self.kind == 'gcn':
            hw = self.lin(h)
            return G.agg(hw) + G.self_w.unsqueeze(1) * hw + self.bias
        return G.agg(self.lin(h)) + self.root(h)

    def neighbours(self, G, h_stage1):                          # aggregated messages from stage-one reps
        return G.agg(self.lin(h_stage1))

    def centre_pair(self, h_mix, sw_i, sw_j):                   # self terms for branch i and branch j
        if self.kind == 'gcn':
            hw = self.lin(h_mix)
            return sw_i.unsqueeze(1) * hw + self.bias, sw_j.unsqueeze(1) * hw + self.bias
        c = self.root(h_mix)
        return c, c


class TwoStageMixupGNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, num_layers, kind='gcn', jk=None, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList([MixConv(in_dim if l == 0 else hidden, hidden, kind) for l in range(num_layers)])
        self.kind, self.jk, self.dropout = kind, jk, dropout
        self.cls = nn.Linear(hidden * num_layers if jk == 'cat' else hidden, out_dim)

    def _post(self, h):
        return F.dropout(F.relu(h), self.dropout, self.training)

    def _readout(self, hs):
        if self.jk == 'cat':
            h = torch.cat(hs, dim=1)
        elif self.jk == 'max':
            h = torch.stack(hs, 0).max(0).values
        else:
            h = hs[-1]
        return self.cls(h)

    def forward(self, x, G):
        hs, h = [], x
        for conv in self.convs:
            h = self._post(conv(G, h))
            hs.append(h)
        return self._readout(hs)

    def forward_mixup(self, x, G, lam, perm):
        # ---- Stage 1: feed-forward without Mixup (Alg. 1, lines 1-8) ----
        stage1, h = [x], x
        for conv in self.convs[:-1]:
            h = self._post(conv(G, h))
            stage1.append(h)
        # ---- Stage 2: two-branch Mixup graph convolutions (lines 9-20) ----
        h_mix = lam * x + (1 - lam) * x[perm]                   # Eq. (5)
        sw_i, sw_j = G.self_w, G.self_w[perm]
        outs = []
        for l, conv in enumerate(self.convs):
            nb = conv.neighbours(G, stage1[l])                   # messages from stage-one neighbours
            c_i, c_j = conv.centre_pair(h_mix, sw_i, sw_j)
            h_i = F.relu(nb + c_i)                               # branch on node i's topology   (Eq. 6)
            h_j = F.relu(nb[perm] + c_j)                         # branch on node j's topology   (Eq. 6)
            h_mix = F.dropout(lam * h_i + (1 - lam) * h_j, self.dropout, self.training)   # Eq. (7)
            outs.append(h_mix)
        return self._readout(outs)


class GATNet(nn.Module):
    """2-layer GAT baseline (Velickovic et al., 2018) for Table 3."""
    def __init__(self, in_dim, out_dim, hidden=8, heads=8, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.c1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
        self.c2 = GATConv(hidden * heads, out_dim, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = F.dropout(x, self.dropout, self.training)
        x = F.elu(self.c1(x, edge_index))
        x = F.dropout(x, self.dropout, self.training)
        return self.c2(x, edge_index)

print('models ready')

models ready


In [ ]:
class ResultStore:
    def __init__(self, path):
        self.path, self.rows = path, {}
        if osp.exists(path):
            with open(path) as f:
                for line in f:
                    if line.strip():
                        r = json.loads(line)
                        self.rows[r['key']] = r

    def __contains__(self, key):
        return key in self.rows

    def put(self, key, rec):
        rec = dict(rec, key=key)
        self.rows[key] = rec
        with open(self.path, 'a') as f:
            f.write(json.dumps(rec) + '\n')

    def frame(self, prefix):
        return pd.DataFrame([r for k, r in self.rows.items() if k.startswith(prefix)])


STORE = ResultStore(osp.join(ROOT, 'paper_graph_results.jsonl'))


def make_split(n, split_id, train=0.6, val=0.2):
    rng = np.random.RandomState(1000 + split_id)
    perm = rng.permutation(n)
    a, b = int(train * n), int((train + val) * n)
    return {k: torch.from_numpy(v).long() for k, v in
            {'train': perm[:a], 'val': perm[a:b], 'test': perm[b:]}.items()}


_saved = torch.load(osp.join(ROOT, 'paper_graph.pt'))
_g = _saved[GRAPH_VARIANT]
_ei = _g['edge_index']
_ei = torch.cat([_ei, _ei.flip(0)], dim=1)               # undirected
_keep = _g['y'] >= 0
PAPER_GRAPH = SimpleNamespace(name='PaperGraph', x=_g['x'], y=_g['y'], edge_index=_ei, n=_g['x'].size(0),
                              num_features=_g['x'].size(1), num_classes=len(_g['classes']), graphs={})
print(f"loaded '{GRAPH_VARIANT}' graph: {PAPER_GRAPH.n} nodes, {_ei.size(1) // 2} edges, "
      f"{PAPER_GRAPH.num_classes} classes, features {tuple(PAPER_GRAPH.x.shape)} via {_saved['feature_method']}")

loaded 'main' graph: 2813 nodes, 19936 edges, 6 classes, features (2813, 384) via sentence-transformers/all-MiniLM-L6-v2


In [ ]:
# (copied unchanged from the reproduction notebook)
T3_SPECS = {
    'GCN':            dict(arch='mix', kind=T3_CONV, jk=None,  mixup=False, dropedge=False),
    'GAT':            dict(arch='gat', fixed_depth=2),
    'JKNet':          dict(arch='mix', kind='gcn',   jk='cat', mixup=False, dropedge=False),
    'DropEdge+GCN':   dict(arch='mix', kind=T3_CONV, jk=None,  mixup=False, dropedge=True),
    'DropEdge+JKNet': dict(arch='mix', kind='gcn',   jk='cat', mixup=False, dropedge=True),
    'Mixup+GCN':      dict(arch='mix', kind=T3_CONV, jk=None,  mixup=True,  dropedge=False),
    'Mixup+JKNet':    dict(arch='mix', kind='gcn',   jk='cat', mixup=True,  dropedge=False),
    # authors' repository configuration (Section 7): GraphConv layer, 3 layers, Beta(4, 4)
    'Repo:GraphConv':       dict(arch='mix', kind='graphconv', jk=None, mixup=False, dropedge=False, fixed_depth=3),
    'Repo:Mixup+GraphConv': dict(arch='mix', kind='graphconv', jk=None, mixup=True,  dropedge=False, fixed_depth=3, alpha=4.0),
}


def full_graph(D, kind):
    if kind not in D.graphs:
        D.graphs[kind] = make_graph(D.rowptr, D.col, D.n, kind, DEVICE)
    return D.graphs[kind]


def train_transductive(D, split, method, depth, seed, hp=None):
    spec = T3_SPECS[method]
    set_seed(seed)
    x, y = D.x.to(DEVICE), D.y.to(DEVICE)
    tr, va, te = (split[k].to(DEVICE) for k in ('train', 'val', 'test'))

    if spec['arch'] == 'gat':
        hp = hp or GAT_HP
        model = GATNet(D.num_features, D.num_classes, hp['hidden'], hp['heads'], hp['dropout']).to(DEVICE)
        ei = D.edge_index.to(DEVICE)
        fwd_eval = lambda: model(x, ei)
    else:
        hp = hp or T3_HP
        model = TwoStageMixupGNN(D.num_features, hp['hidden'], D.num_classes, depth,
                                 spec['kind'], spec['jk'], hp['dropout']).to(DEVICE)
        G_full = full_graph(D, spec['kind'])
        fwd_eval = lambda: model(x, G_full)
    opt = torch.optim.Adam(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])
    alpha = spec.get('alpha', MIXUP_ALPHA)

    best = dict(val=-1.0, val_loss=float('inf'), test=0.0, epoch=0)
    t0 = time.time()
    for epoch in range(1, hp['epochs'] + 1):
        model.train()
        opt.zero_grad()
        if spec['arch'] == 'gat':
            out = model(x, ei)
            loss = F.cross_entropy(out[tr], y[tr])
        else:
            G = G_full
            if spec['dropedge']:
                rp, cl = edge_index_to_csr(drop_edge_undirected(D.edge_index, DROPEDGE_P), D.n)
                G = make_graph(rp, cl, D.n, spec['kind'], DEVICE)
            if spec['mixup']:
                lam = float(np.random.beta(alpha, alpha))
                perm = torch.arange(D.n, device=DEVICE)
                perm[tr] = tr[torch.randperm(tr.numel(), device=DEVICE)]
                out = model.forward_mixup(x, G, lam, perm)
                loss = lam * F.cross_entropy(out[tr], y[tr]) + (1 - lam) * F.cross_entropy(out[tr], y[perm[tr]])
            else:
                out = model(x, G)
                loss = F.cross_entropy(out[tr], y[tr])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            out = fwd_eval()
            pred = out.argmax(1)
            val_acc = (pred[va] == y[va]).float().mean().item()
            test_acc = (pred[te] == y[te]).float().mean().item()
            val_loss = F.cross_entropy(out[va], y[va]).item()
        if val_acc > best['val'] or (val_acc == best['val'] and val_loss < best['val_loss']):
            best = dict(val=val_acc, val_loss=val_loss, test=test_acc, epoch=epoch)
    return dict(val=100 * best['val'], test=100 * best['test'], test_last=100 * test_acc,
                best_epoch=best['epoch'], secs=round(time.time() - t0, 2))

print('Table 3 training loop ready')

Table 3 training loop ready


In [ ]:
PAPER_GRAPH.rowptr, PAPER_GRAPH.col = edge_index_to_csr(PAPER_GRAPH.edge_index, PAPER_GRAPH.n)
T3_SPECS['GraphConv'] = dict(arch='mix', kind='graphconv', jk=None, mixup=False, dropedge=False)
T3_SPECS['Mixup+GraphConv a4'] = dict(arch='mix', kind='graphconv', jk=None, mixup=True, dropedge=False, alpha=4.0)
load_planetoid = lambda name: PAPER_GRAPH               # the shared runner loads datasets through this name

In [ ]:
# (copied unchanged from the reproduction notebook)
def run_transductive_grid(tag, datasets, methods, depths, n_splits, n_seeds, hp=None):
    jobs = []
    for name in datasets:
        for m in methods:
            spec = T3_SPECS[m]
            for L in ([spec['fixed_depth']] if 'fixed_depth' in spec else depths):
                for s in range(n_splits):
                    for t in range(n_seeds):
                        key = f'{tag}|{name}|{m}|L{L}|s{s}|t{t}'
                        if key not in STORE:
                            jobs.append((key, name, m, L, s, t))
    print(f'{tag}: {len(jobs)} runs to do')
    for key, name, m, L, s, t in tqdm(jobs, desc=tag):
        D = load_planetoid(name)
        rec = train_transductive(D, make_split(D.n, s), m, L, seed=100 * s + t, hp=hp)
        STORE.put(key, dict(rec, table=tag, dataset=name, method=m, depth=L, split=s, trial=t))


def summarise(tag, metric='test'):
    """Mean ± std over runs; the depth is chosen per (dataset, method) by mean validation score."""
    df = STORE.frame(tag + '|')
    if df.empty:
        return df, df
    g = (df.groupby(['dataset', 'method', 'depth'])
           .agg(val=('val', 'mean'), mean=(metric, 'mean'), std=(metric, 'std'), runs=(metric, 'size'))
           .reset_index())
    g['std'] = g['std'].fillna(0.0)
    best = g.loc[g.groupby(['dataset', 'method'])['val'].idxmax()].reset_index(drop=True)
    return best, g

### 9.1 Run

In [ ]:
run_transductive_grid('PG', ['PaperGraph'], NEW_METHODS, NEW_DEPTHS, NEW_SPLITS, NEW_SEEDS)

PG: 160 runs to do


PG:   0%|          | 0/160 [00:00<?, ?it/s]

### 9.2 Results and Mixup gain (paired over the same splits and seeds)

In [ ]:
from scipy import stats

best_pg, all_pg = summarise('PG')
res = best_pg.assign(result=lambda d: d['mean'].round(1).astype(str) + ' ± ' + d['std'].round(1).astype(str))
display(res[['method', 'depth', 'result', 'runs']].set_index('method').loc[[m for m in NEW_METHODS if m in set(res.method)]])

df = STORE.frame('PG|')
rows = []
for base, mix in [('GCN', 'Mixup+GCN'), ('GraphConv', 'Mixup+GraphConv a4')]:
    if not {base, mix} <= set(best_pg.method):
        continue
    pick = lambda m: int(best_pg[best_pg.method == m].depth.iloc[0])
    b = df[(df.method == base) & (df.depth == pick(base))].set_index(['split', 'trial'])
    m = df[(df.method == mix) & (df.depth == pick(mix))].set_index(['split', 'trial'])
    idx = b.index.intersection(m.index)
    diff = m.loc[idx, 'test'] - b.loc[idx, 'test']
    rows.append({'Comparison': f'{mix} − {base}', 'Backbone': round(b.loc[idx, 'test'].mean(), 2),
                 'Mixup': round(m.loc[idx, 'test'].mean(), 2), 'Gain': round(diff.mean(), 2),
                 'Mixup wins': f'{int((diff > 0).sum())}/{len(diff)}',
                 'p-value (paired t)': round(stats.ttest_rel(m.loc[idx, 'test'], b.loc[idx, 'test']).pvalue, 3)})
pg_gain = pd.DataFrame(rows)
display(pg_gain)
if not all_pg.empty:
    display(all_pg.pivot_table(index='method', columns='depth', values='mean').round(1))
best_pg.to_csv(osp.join(ROOT, 'paper_graph_results_summary.csv'), index=False)
pg_gain.to_csv(osp.join(ROOT, 'paper_graph_mixup_gain.csv'), index=False)
print('Reference – reproduction findings: GCN α=1 gain ≈ 0 on Cora/Citeseer/Pubmed; GraphConv α=4 gain +1.5 Cora, +0.5 Pubmed')

,depth,result,runs
method,,,
GCN,2,82.8 ± 1.5,20
Mixup+GCN,2,82.6 ± 1.3,20
GraphConv,2,85.5 ± 1.1,20
Mixup+GraphConv a4,2,85.3 ± 1.0,20


,Comparison,Backbone,Mixup,Gain,Mixup wins,p-value (paired t)
0,Mixup+GCN − GCN,82.80,82.61,-0.19,7/20,0.284
1,Mixup+GraphConv a4 − GraphConv,85.46,85.30,-0.16,7/20,0.515


depth,2,3
method,,
GCN,82.8,81.9
GraphConv,85.5,85.0
Mixup+GCN,82.6,81.6
Mixup+GraphConv a4,85.3,85.0


Reference – reproduction findings: GCN α=1 gain ≈ 0 on Cora/Citeseer/Pubmed; GraphConv α=4 gain +1.5 Cora, +0.5 Pubmed
